In [1]:
try:
    import subprocess, sys, os

    wheel_path = "../input/weights/timm-0.3.1-py3-none-any.whl"
    if os.path.isfile(wheel_path):
        subprocess.check_call([sys.executable, "-m", "pip", "install", wheel_path])
except Exception:
    pass




In [2]:
import os
import random
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.parameter import Parameter
import torchvision.transforms as transforms
from torchvision.transforms import functional as FT
from torch.utils.data import Dataset, DataLoader, random_split, TensorDataset
from PIL import Image, ImageChops
import timm




/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True




In [4]:
threshold = [0.75, 1.5, 2.5, 3.5]


def regress2class(out):
    prediction = torch.zeros(out.size(0), device=out.device)
    for i in range(4):
        prediction += (out.data >= threshold[i]).squeeze()
    return prediction


def ordinal2class_prob(out):
    pred_prob = torch.zeros(out.size(0), 5, device=out.device)
    pred_prob[:, 0] = (1 - out[:, 0]).squeeze()
    pred_prob[:, 1] = (out[:, 0] * (1 - out[:, 1])).squeeze()
    pred_prob[:, 2] = (out[:, 1] * (1 - out[:, 2])).squeeze()
    pred_prob[:, 3] = (out[:, 2] * (1 - out[:, 3])).squeeze()
    pred_prob[:, 4] = out[:, 3].squeeze()
    return F.softmax(pred_prob, dim=1)


def regress2class_prob(out):
    pred_prob = torch.zeros((out.size(0), 5), device=out.device)
    for i in range(out.size(0)):
        if out[i] < 4.0:
            l1 = int(math.floor(out[i].item()))
            l2 = int(math.ceil(out[i].item()))
            pred_prob[i][l1] = 1 - (out[i] - l1)
            pred_prob[i][l2] = 1 - (l2 - out[i])
        else:
            pred_prob[i][4] = 1.0
    return pred_prob




In [5]:
def gem(x, p=3, eps=1e-6):
    return F.avg_pool2d(x.clamp(min=eps).pow(p), (x.size(-2), x.size(-1))).pow(1.0 / p)


class GeM(nn.Module):
    def __init__(self, p=3, eps=1e-6, flatten=False):
        super(GeM, self).__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps
        self.flatten = flatten

    def forward(self, x):
        x = gem(x, p=self.p, eps=self.eps)
        if self.flatten:
            x = x.flatten(1)
        return x

    def __repr__(self):
        return f"{self.__class__.__name__}(p={self.p.item():.4f}, eps={self.eps})"


class ThreeStage_Model(nn.Module):
    def __init__(self, backbone=None):
        super(ThreeStage_Model, self).__init__()
        self.backbone = timm.create_model(
            "tf_efficientnet_b4_ns", pretrained=True, num_classes=0
        )
        self.backbone.global_pool = GeM(flatten=True)

        feat_dim = self.backbone.num_features

        self.classifier = nn.Sequential(
            nn.SiLU(),
            nn.Linear(feat_dim, 500),
            nn.SiLU(),
            nn.Linear(500, 5),
        )
        self.regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(feat_dim, 500),
            nn.SiLU(),
            nn.Linear(500, 1),
        )
        self.ordinal = nn.Sequential(
            nn.SiLU(),
            nn.Linear(feat_dim, 500),
            nn.SiLU(),
            nn.Linear(500, 4),
        )
        self.final_regressor = nn.Sequential(
            nn.SiLU(),
            nn.Linear(10, 1),
        )

    def forward(self, x, final=False):
        x = self.backbone(x)
        c_out = self.classifier(x)
        r_out = self.regressor(x)
        o_out = self.ordinal(x)
        if final:
            out = torch.cat((c_out, r_out, o_out), 1)
            out = self.final_regressor(out)
            out = torch.sigmoid(out) * 4.5
            return out
        else:
            r_out = torch.sigmoid(r_out) * 4.5
            o_out = torch.sigmoid(o_out)
            return c_out, r_out, o_out




In [6]:
class photometric_distort(object):
    def __call__(self, image):
        distortions = [
            FT.adjust_brightness,
            FT.adjust_contrast,
            FT.adjust_saturation,
            FT.adjust_hue,
        ]
        random.shuffle(distortions)
        for d in distortions:
            if random.random() < 0.5:
                if d.__name__ == "adjust_hue":
                    adjust_factor = random.uniform(-16 / 255.0, 16 / 255.0)
                else:
                    adjust_factor = random.uniform(0.7, 1.3)
                image = d(image, adjust_factor)
        return image


class cropTo4_3(object):
    def __call__(self, image):
        w, h = image.size
        if (w / h) >= (4 / 3):
            new_h = h
            new_w = int(h * 4 / 3)
        else:
            new_h = int(w * 3 / 4)
            new_w = w
        left = (w - new_w) / 2
        top = (h - new_h) / 2
        right = left + new_w
        bottom = top + new_h
        return image.crop((left, top, right, bottom))


class trim(object):
    def __call__(self, image):
        bg = Image.new(image.mode, image.size, image.getpixel((0, 0)))
        diff = ImageChops.difference(image, bg)
        diff = ImageChops.add(diff, diff, 2.0, -10)
        bbox = diff.getbbox()
        if bbox:
            return image.crop(bbox)
        return image




In [7]:
class AptosDataset(Dataset):
    def __init__(self, df, img_dir, transform):
        self.ids = df["id_code"].values
        self.labels = df["diagnosis"].values if "diagnosis" in df.columns else None
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        id_code = self.ids[idx]
        img_path = os.path.join(self.img_dir, f"{id_code}.png")
        image = Image.open(img_path).convert("RGB")
        image = self.transform(image)
        if self.labels is not None:
            label = int(self.labels[idx])
            return image, label
        else:
            return image, id_code


input_size = 380
train_transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        photometric_distort(),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

val_transform = transforms.Compose(
    [
        trim(),
        cropTo4_3(),
        transforms.Resize((input_size * 3 // 4, input_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.384, 0.258, 0.174], std=[0.124, 0.089, 0.094]),
    ]
)

train_df = pd.read_csv("../input/aptos2019-blindness-detection/train.csv")
train_img_dir = "../input/aptos2019-blindness-detection/train_images"

full_dataset = AptosDataset(train_df, train_img_dir, train_transform)
val_size = int(0.1 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

num_workers = min(8, os.cpu_count() or 2)




In [8]:
net = ThreeStage_Model()
weight_path = "../input/weights/B4_3stage_58epoch_CLAHE.pkl"
if os.path.isfile(weight_path):
    try:
        state_dict = torch.load(weight_path, map_location=device)
        net.load_state_dict(state_dict, strict=True)
        print("Pretrained weights loaded.")
    except Exception as e:
        print(f"Warning: failed to load weights ({e}); using random init.")
else:
    print("Weight file not found; proceeding with random initialization.")

net = net.to(device)

for param in net.backbone.parameters():
    param.requires_grad = False

if hasattr(net.backbone, "blocks"):
    for param in net.backbone.blocks[-1].parameters():
        param.requires_grad = True
else:
    for name, param in net.backbone.named_parameters():
        if "conv_head" in name:
            param.requires_grad = True

net.eval()
with torch.no_grad():

    def extract_features(dataset):
        loader = DataLoader(
            dataset,
            batch_size=32,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=True,
            persistent_workers=True,
        )
        feats = []
        labs = []
        for imgs, labels in loader:
            imgs = imgs.to(device)
            f = net.backbone(imgs)  # frozen (mostly) backbone output
            feats.append(f.cpu())
            labs.append(labels)
        return torch.cat(feats, dim=0), torch.cat(labs, dim=0)

    train_feats, train_labels = extract_features(train_dataset)
    val_feats, val_labels = extract_features(val_dataset)

train_feat_dataset = TensorDataset(train_feats, train_labels)
val_feat_dataset = TensorDataset(val_feats, val_labels)

train_loader = DataLoader(
    train_feat_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)
val_loader = DataLoader(
    val_feat_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

class_counts = torch.bincount(train_labels)
class_weights = 1.0 / (class_counts.float() + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_counts)
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, net.parameters()), lr=1e-3
)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

epochs = 45
best_val_acc = -1.0
best_state_dict = None

net.train()
for epoch in range(epochs):
    running_loss = 0.0
    for feats, labels in train_loader:
        feats = feats.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        c_out = net.classifier(feats)  # only the classifier head
        loss = criterion(c_out, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * feats.size(0)

    epoch_loss = running_loss / train_size

    net.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats = feats.to(device)
            labels = labels.to(device)
            c_out = net.classifier(feats)
            preds = torch.argmax(c_out, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    val_acc = correct / total if total > 0 else 0
    print(f"Epoch {epoch+1}/{epochs} - Loss: {epoch_loss:.4f} - Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state_dict = net.state_dict()

    scheduler.step()
    net.train()

if best_state_dict is not None:
    net.load_state_dict(best_state_dict)
    print(f"Best model restored (Val Acc = {best_val_acc:.4f})")
net.eval()




/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:138: UserWarning: Mapping deprecated model name tf_efficientnet_b4_ns to current tf_efficientnet_b4.ns_jft_in1k.
  model = create_fn(


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

Weight file not found; proceeding with random initialization.


/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch 1/45 - Loss: 5.7325 - Val Acc: 0.1611
Epoch 2/45 - Loss: 2.0093 - Val Acc: 0.3465
Epoch 3/45 - Loss: 1.5858 - Val Acc: 0.2857
Epoch 4/45 - Loss: 1.4414 - Val Acc: 0.4529
Epoch 5/45 - Loss: 1.4295 - Val Acc: 0.3526
Epoch 6/45 - Loss: 1.5247 - Val Acc: 0.4742
Epoch 7/45 - Loss: 1.5471 - Val Acc: 0.4043
Epoch 8/45 - Loss: 1.3464 - Val Acc: 0.2371
Epoch 9/45 - Loss: 1.2816 - Val Acc: 0.4620
Epoch 10/45 - Loss: 1.2344 - Val Acc: 0.4529
Epoch 11/45 - Loss: 1.1836 - Val Acc: 0.5198
Epoch 12/45 - Loss: 1.1595 - Val Acc: 0.5167
Epoch 13/45 - Loss: 1.1382 - Val Acc: 0.5289
Epoch 14/45 - Loss: 1.1138 - Val Acc: 0.5076
Epoch 15/45 - Loss: 1.0915 - Val Acc: 0.4894
Epoch 16/45 - Loss: 1.0609 - Val Acc: 0.5258
Epoch 17/45 - Loss: 1.0499 - Val Acc: 0.5258
Epoch 18/45 - Loss: 1.0406 - Val Acc: 0.5258
Epoch 19/45 - Loss: 1.0314 - Val Acc: 0.5471
Epoch 20/45 - Loss: 1.0103 - Val Acc: 0.5350
Epoch 21/45 - Loss: 1.0028 - Val Acc: 0.5380
Epoch 22/45 - Loss: 0.9905 - Val Acc: 0.5380
Epoch 23/45 - Loss:

ThreeStage_Model(
  (backbone): EfficientNet(
    (conv_stem): Conv2dSame(3, 48, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn1): BatchNormAct2d(
      48, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): SiLU(inplace=True)
    )
    (blocks): Sequential(
      (0): Sequential(
        (0): DepthwiseSeparableConv(
          (conv_dw): Conv2d(48, 48, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=48, bias=False)
          (bn1): BatchNormAct2d(
            48, eps=0.001, momentum=0.1, affine=True, track_running_stats=True
            (drop): Identity()
            (act): SiLU(inplace=True)
          )
          (aa): Identity()
          (se): SqueezeExcite(
            (conv_reduce): Conv2d(48, 12, kernel_size=(1, 1), stride=(1, 1))
            (act1): SiLU(inplace=True)
            (conv_expand): Conv2d(12, 48, kernel_size=(1, 1), stride=(1, 1))
            (gate): Sigmoid()
          )
          (conv_pw): Conv2d

In [9]:
test_df = pd.read_csv("../input/aptos2019-blindness-detection/test.csv")
test_img_dir = "../input/aptos2019-blindness-detection/test_images"

test_dataset = AptosDataset(test_df, test_img_dir, val_transform)


def collate_fn(batch):
    imgs = torch.stack([item[0] for item in batch])
    ids = [item[1] for item in batch]
    return imgs, ids


test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    persistent_workers=True,
    collate_fn=collate_fn,
)

submission = []
net.zero_grad()
with torch.no_grad():
    for imgs, ids in test_loader:
        imgs = imgs.to(device)

        # ----- original orientation -----
        c_out, _, o_out = net(imgs, final=False)
        probs_c = F.softmax(c_out, dim=1)
        probs_o = ordinal2class_prob(o_out)
        probs_orig = (probs_c + probs_o) / 2.0

        # ----- horizontal flip -----
        imgs_h = torch.flip(imgs, dims=[3])
        c_out_h, _, o_out_h = net(imgs_h, final=False)
        probs_c_h = F.softmax(c_out_h, dim=1)
        probs_o_h = ordinal2class_prob(o_out_h)
        probs_h = (probs_c_h + probs_o_h) / 2.0

        # ----- vertical flip -----
        imgs_v = torch.flip(imgs, dims=[2])
        c_out_v, _, o_out_v = net(imgs_v, final=False)
        probs_c_v = F.softmax(c_out_v, dim=1)
        probs_o_v = ordinal2class_prob(o_out_v)
        probs_v = (probs_c_v + probs_o_v) / 2.0

        # Average over augmentations
        avg_probs = (probs_orig + probs_h + probs_v) / 3.0
        preds = torch.argmax(avg_probs, dim=1).cpu().numpy()

        for idx, pred in zip(ids, preds):
            submission.append([idx, int(pred)])

submission = np.array(submission)

if len(submission) == 0:
    raise RuntimeError("Submission array is empty; check data loading.")
df = pd.DataFrame(submission, columns=["id_code", "diagnosis"])
df.to_csv("submission.csv", index=False)

/usr/local/lib/python3.11/dist-packages/torch/utils/data/dataloader.py:624: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
